<a href="https://colab.research.google.com/github/cactus1386/NationalCard-ImageProccessing/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and import Libraries

In [1]:
! pip install ultralytics easyocr

  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached ultralytics_thop-2.0.14-py3-none-any.whl.metadata (9.4 kB)
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 4.6 MB/s eta 0:00:00
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached ultralytics_thop-2.0.14-py3-none-any.whl (26 kB)
Using cached py_cpuinfo-9.0.0-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
# from google.colab.patches import cv2_imshow
import os
import pandas as pd

# Set model and path

In [5]:
objects_model = YOLO('TextDetection.pt') # set yolo model for object detection
card_model = YOLO('CardDetection.pt') # set yolo model for card detection
img_path = './test_image_phase1/2.jpg' # set image path
ocr = easyocr.Reader(['fa']) # set persian ocr reader

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [6]:
img_path = './test_image_phase1/2.jpg'

# Crop Card

In [7]:
def crop_card(path):
  img = cv2.imread(path) # read image

  results = card_model(img) # set card model for image

  for result in results:
    boxes = result.boxes
    for box in boxes:
      xyxy = box.xyxy[0]
      x1, y1, x2, y2 = map(int, xyxy.tolist()) # convert to int and set x and y

      # crop image and show that
      cropped_img = img[y1:y2, x1:x2]
      # cv2_imshow(cropped_img)
      return cropped_img

# Make crop image

In [8]:
crop = crop_card(img_path)


0: 288x640 1 Card, 258.0ms
Speed: 15.4ms preprocess, 258.0ms inference, 20.7ms postprocess per image at shape (1, 3, 288, 640)


# Make function for crop and read text parts

In [9]:
def process_img(img):
  ignore_class = ['FatherName', 'LastName', 'Name']
  data = {}
  results = objects_model(img) # set model for image
  for result in results:
    boxes = result.boxes
    for box in boxes:
      xyxy = box.xyxy[0]
      x1, y1, x2, y2 = map(int, xyxy.tolist()) # convert to int and set x and y

      label = result.names[int(box.cls)] # get class name

      if label in ignore_class: # ignore labels in ignore class
          continue

      # crop image and show that
      cropped_img = img[(y1 + 7):(y2 + 7), (x1 + 7):(x2 + 7)]
      # cv2_imshow(cropped_img)
      # set ocr and read text on the cropped image
      ocr_result = ocr.readtext(cropped_img)

      # OCR result
      for (bbox, text, conf) in ocr_result:
        print(f"Text: {text}, Confidence: {conf}")
        data[label] = text

  return data
          # draw box
        # cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
      # cv2_imshow(img)

# Call function and use it

In [14]:
import re

# تابع بازسازی تاریخ از متن OCR
def parse_date_from_text(text):
    # حذف فاصله‌ها و کاراکترهای غیر ضروری
    text = text.replace(' ', '').replace('|', '').strip()
    
    # استخراج همه اعداد از متن
    numbers = re.findall(r'\d+', text)  # پیدا کردن اعداد
    
    if len(numbers) < 3:
        # اگر کمتر از ۳ عدد پیدا شود، تاریخ ناقص است
        print(f"Invalid date: {text}")
        return None
    
    # تلاش برای تشخیص سال، ماه و روز
    year, month, day = None, None, None

    # پیدا کردن سال (باید 4 رقم باشد)
    for num in numbers:
        if len(num) == 4 and int(num) > 1300:  # فرض می‌کنیم تاریخ شمسی است
            year = num
            numbers.remove(num)
            break

    # پیدا کردن ماه و روز
    if len(numbers) >= 2:
        month, day = numbers[:2]
    
    # اگر تاریخ کامل باشد، به فرمت استاندارد برمی‌گردیم
    if year and month and day:
        return f"{year}/{month}/{day}"
    else:
        print(f"Incomplete date: {text}")
        return None

In [15]:
# بخش detect بهبود یافته
def detect(folder):
    detected = []
    
    for img in os.listdir(folder):
        print(img)
        # Only process specific image formats
        if img.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.heic')):
            img_path = os.path.join(folder, img)
            
            # Initialize the dictionary for extracted data
            data = {
                'image_id': '', 
                'national_id': '', 
                'birth_year': '',
                'birth_month': '', 
                'birth_day': '', 
                'expiry_year': '',
                'expiry_month': '', 
                'expiry_day': ''
            }
            
            data['image_id'] = img.split('.')[0]  # Extract image_id from the image name
            
            card = crop_card(img_path)  # Get cropped image of the card
            extracted_data = process_img(card)  # Extract information from the card
            
            # Iterate through extracted data and map it to the dictionary
            for key, value in extracted_data.items():
                value = value.replace(' ', '')  # Remove spaces for clean data
                
                if key == 'Expire':
                    try:
                        # استفاده از تابع parse_date_from_text
                        parsed_date = parse_date_from_text(value)
                        if parsed_date:
                            y, m, d = parsed_date.split('/')
                            data['expiry_year'] = int(y)
                            data['expiry_month'] = int(m)
                            data['expiry_day'] = int(d)
                    except ValueError:
                        print(f"Invalid expiry date format for {data['image_id']}: {value}")
                        pass
                
                if key == 'Birth':
                    try:
                        # استفاده از تابع parse_date_from_text
                        parsed_date = parse_date_from_text(value)
                        if parsed_date:
                            y, m, d = parsed_date.split('/')
                            data['birth_year'] = int(y)
                            data['birth_month'] = int(m)
                            data['birth_day'] = int(d)
                    except ValueError:
                        print(f"Invalid birth date format for {data['image_id']}: {value}")
                        pass
                
                if key == 'National':
                    data['national_id'] = value  # Store national ID
                
                # Add more keys here as needed (if you want to extract more fields)

            detected.append(data)  # Append the data dictionary to the detected list

    # Convert the list of dictionaries to a DataFrame and save as CSV
    df = pd.DataFrame(detected)
    df.to_csv('image_phase1.csv', index=False, encoding='utf-8')
    print("CSV file has been created: image_phase1.csv")

In [16]:
detect('./test')

10.jpg



0: 320x640 1 Card, 107.3ms
Speed: 3.3ms preprocess, 107.3ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 640)

0: 352x640 1 Birth, 1 Expire, 1 FatherName, 1 LastName, 1 Name, 1 National, 160.4ms
Speed: 1.6ms preprocess, 160.4ms inference, 2.1ms postprocess per image at shape (1, 3, 352, 640)
Text: ٤٥١٨٧٥٥٣٢٧, Confidence: 0.9937506079019558
Text: ٩/٥٧/١٩ ١٤٥, Confidence: 0.6151200410220207
Text: ٥ ٣٧٩/١١/٢, Confidence: 0.7080627648201291
Incomplete date: ٩/٥٧/١٩١٤٥
11.jpg

0: 640x384 1 Card, 114.1ms
Speed: 3.0ms preprocess, 114.1ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x576 2 Expires, 1 National, 181.9ms
Speed: 3.5ms preprocess, 181.9ms inference, 3.2ms postprocess per image at shape (1, 3, 640, 576)
Text: :, Confidence: 0.12486916401084702
Text: ؟, Confidence: 0.3753460563826252
Text: ٤, Confidence: 0.5161827496999116
Text: ٍ, Confidence: 0.11710935312117954
Text: امر , Confidence: 0.4582430134676573
Text: ج, Confidence: 0.04439434152616